In [9]:
import cv2
import numpy as np
import os

def unstack_colors_with_tolerance(image_path, output_dir, tolerance=30, min_height=2):
    stacked_colors = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
    if stacked_colors is None:
        print(f"Could not load {image_path}")
        return
        
    os.makedirs(output_dir, exist_ok=True)
    height, width = stacked_colors.shape[:2]
    mid_x = width // 2

    cut_points = [0]
    
    # Grab the first pixel as our baseline, cast to int32 to avoid overflow during math
    base_color = stacked_colors[0, mid_x].astype(np.int32)

    for y in range(1, height):
        pixel_color = stacked_colors[y, mid_x].astype(np.int32)
        
        # Calculate the mathematical distance between the current pixel and the base color
        # This handles RGBA channels simultaneously
        color_diff = np.linalg.norm(pixel_color - base_color)
        
        # If the color difference exceeds our threshold, it's a totally new strip
        if color_diff > tolerance:
            cut_points.append(y)
            base_color = pixel_color # Set the new color as the baseline

    cut_points.append(height)

    num_strips = len(cut_points) - 1
    valid_strips = 0

    print(f"Detected {num_strips} potential color changes.")

    for i in range(num_strips):
        y_start = cut_points[i]
        y_end = cut_points[i+1]
        
        strip_height = y_end - y_start
        
        # Ignore 1px or 2px thick "separator" lines or anti-aliasing artifacts
        if strip_height >= min_height:
            color_strip = stacked_colors[y_start:y_end, :]
            
            # Save the valid strip
            cv2.imwrite(os.path.join(output_dir, f'color_strip_{valid_strips:02d}.png'), color_strip)
            valid_strips += 1

    print(f"Successfully extracted {valid_strips} usable color fills!")

if __name__ == "__main__":
    unstack_colors_with_tolerance('sprite_075.png', 'unstacked_colors')

Detected 12 potential color changes.
Successfully extracted 12 usable color fills!
